<a href="https://colab.research.google.com/github/adreno660/Torrent-To-Google-Drive-Downloader/blob/master/v5_Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [6]:
print("Uninstalling potentially incompatible packages...")
!apt-get remove python3-libtorrent --yes
!pip uninstall lbry-libtorrent --yes
print("Installing the correct version of libtorrent...")
!pip install libtorrent

Uninstalling potentially incompatible packages...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'python3-libtorrent' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Installing the correct version of libtorrent...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 83.0 MB/s eta 0:00:00


In [7]:
import libtorrent as lt
ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []
print("libtorrent version:", lt.version)
print("\nSuccess! Libtorrent is now correctly imported and working.")

libtorrent version: 2.0.11.0

Success! Libtorrent is now correctly imported and working.


/tmp/ipykernel_8889/2533598611.py:3: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


### Mount Google Drive
To stream files we need to mount Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [8]:
from google.colab import files

# Ensure downloads list and session exist
if 'downloads' not in globals():
    downloads = []
if 'ses' not in globals():
    import libtorrent as lt
    ses = lt.session()
    ses.listen_on(6881, 6891)

source = files.upload()
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

Saving Hytale.Build.18022026-OFME.torrent to Hytale.Build.18022026-OFME.torrent


### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )


### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [10]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}

# Ensure downloads list is initialized if not already defined
if 'downloads' not in globals():
    downloads = []

download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            # Update the description to include downloaded amount and total size
            bar.description = (
                f"{download.name()} "
                f"{s.download_rate / 1000000:.2f} MB/s "
                f"({s.total_downloaded / 1000000:.2f}/{s.total_wanted / 1000000:.2f} MB) "
                f"{state_str[s.state]}"
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            name = download.name() # Get name before removal
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close()
            download_bars.remove(bar)
            print(name, "complete")
    time.sleep(1)

It appears the previous download (`Hytale`) has completed.

In [12]:
# List the contents of the Google Drive torrent directory
!ls -lh "/content/drive/My Drive/Torrent"

total 4.0K
drwxr-xr-x 3 root root 4.0K Mar 16 11:47 Hytale


In [ ]:
if 'ses' in globals() and 'downloads' in globals():
    for download in downloads:
        ses.remove_torrent(download)
    downloads = []
    print('All torrents and magnet links have been cleared.')
else:
    print('No active session or download list found to clear.')

All torrents and magnet links have been cleared.
